# 03 — Data Cleaning

**Day 1, Step 3.** Fix what Step 2 found. Raw files are never modified.

Output goes to `data/processed/` as Parquet — typed, compressed, columnar, and
the actual scalability lever — with a CSV mirror under the filenames the Build
Notes specify.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


In [2]:
from hrai.cleaning.attrition import clean_attrition
from hrai.cleaning.engagement import clean_engagement_events, to_employee_grain
from hrai.cleaning.onet import (clean_essential_skills, clean_occupations,
                                clean_software_skills)

attrition = clean_attrition(load_raw("employee_attrition"))
print("Population A:", attrition.shape)
print("dropped constants:", [c for c in ("EmployeeCount", "Over18", "StandardHours")
                             if c not in attrition.columns])

2026-08-28 01:54:16 | INFO  | raw dataset loaded


2026-08-28 01:54:16 | INFO  | attrition cleaned


Population A: (1470, 35)
dropped constants: ['EmployeeCount', 'Over18', 'StandardHours']


## PII is dropped, not hashed

Nothing downstream needs to re-identify a person, so `FirstName`, `LastName`,
`ADEmail` and `Supervisor` are removed outright. Combined with `nbstripout` in
pre-commit, employee names never reach git history.

In [3]:
events = clean_engagement_events(load_raw("hr_performance_engagement"))
print("event grain:", events.shape)
print("PII columns remaining:",
      [c for c in ("FirstName", "LastName", "ADEmail", "Supervisor") if c in events.columns])

2026-08-28 01:54:16 | INFO  | raw dataset loaded


2026-08-28 01:54:16 | INFO  | engagement events cleaned


event grain: (3150, 36)
PII columns remaining: []


## Grain resolution (finding F5)

3,150 rows over 3,000 employees. The most recent survey wins for point-in-time
attributes; training history is *aggregated*, because a person's training record
is cumulative and taking only the latest row would discard most of it.

In [4]:
employees = to_employee_grain(events)
print(f"{len(events):,} events -> {len(employees):,} employees")
print("unique employee_id:", employees["employee_id"].is_unique)
employees[["employee_id", "Title", "engagement_score", "training_events",
           "training_pass_rate", "is_voluntary_exit"]].head()

2026-08-28 01:54:17 | INFO  | engagement collapsed to employee grain


3,150 events -> 3,000 employees
unique employee_id: True


,employee_id,Title,engagement_score,training_events,training_pass_rate,is_voluntary_exit
0,1001,Software Engineer,2,1,0.0,0
1,1002,Software Engineer,4,1,0.0,0
2,1003,Software Engineer,2,1,0.0,0
3,1004,Software Engineer,3,1,1.0,0
4,1005,Software Engineer,2,1,1.0,0


## Skill-name canonicalisation

The Build Notes call this out directly: 'AWS', 'Amazon Web Services' and
'AWS Cloud' are one skill spelled three ways. Left alone, the gap engine counts
three separate gaps and the org-wide rollup is wrong.

In [5]:
from hrai.cleaning.text import canonical_skill

for raw in ["AWS", "Amazon Web Services", "AWS Cloud",
            "Structured query language SQL", "Microsoft Excel 2016"]:
    print(f"  {raw:32} -> {canonical_skill(raw)}")

  AWS                              -> AWS
  Amazon Web Services              -> AWS
  AWS Cloud                        -> AWS
  Structured query language SQL    -> SQL
  Microsoft Excel 2016             -> Microsoft Excel


## Proving the cleaning worked

Cleaning is judged by one thing: do the checks that failed on raw data now pass
in **strict** mode?

In [6]:
from hrai.validation.runner import validate
from hrai.validation.schemas import attrition_processed_schema, engagement_processed_schema

print(validate(attrition, attrition_processed_schema(), "attrition", mode="strict").summary())
print(validate(employees, engagement_processed_schema(), "engagement", mode="strict").summary())

2026-08-28 01:54:17 | INFO  | schema validation passed


attrition                          PASS
2026-08-28 01:54:17 | INFO  | schema validation passed


engagement                         PASS


Run `make clean-data` to regenerate everything with checksums.